# DART — **Official submission Colab** (one link to rule them all)

> **Submit this file** when the competition asks for a single Colab link. Path: `DART/training/DART_Colab_submission.ipynb` — in Colab: **File → Open notebook → GitHub** and pick that path. (Other `training/*.ipynb` notebooks are **optional** extras: two-bar only, TRL smoke, HTTP demo.)

## Multi-model training + **judge insight figures**

The **simulator** is a weekly diabetes **digital twin** (stochastic patients, **partial** observations at each `step`, multi-term **rubric** reward, safety paths). This notebook both **trains** policies with REINFORCE on **live** `env.step` rollouts and exports **one JSON** plus **two plot batches**: publication curves + a **judge dashboard** (vitals, rubric, action mix, outcome distributions).

**What you are looking at (high level):**
- **Random** vs **small HF LM (distilgpt2)** on the *same* training protocol; optional **4-bit 8B** on GPU; **council (non-LLM)** for diversity.
- **Example trajectories** use the **same env seed** for random vs distil (same simulated patient) so a reviewer can see **only the policy** change; **N-episode** boxplots use **many seeds** to show **variance**.

| Artifact | What it shows |
|----------|----------------|
| `logs/colab_experiment.json` | Episodes, glucose lists, `traces` (1 ep each), `endpoints` (N eps), council repair list |
| `training_curve.png` | Smoothed return vs training order |
| `final_comparison_bars.png` | Tail mean ± std by model |
| `behavior_glucose.png` | FPG vs time (1 ep) |
| `self_repair_episodes.png` | Council return + repair markers |
| `judge_clinical_state.png` | FPG, HbA1c, eGFR, **weekly $** (example ep / policy) |
| `judge_step_and_cumulative_return.png` | Per-step and cumulative return (rubric) |
| `judge_action_mix.png` | Parsed action `type` counts (same example eps) |
| `judge_rubric_episode_totals.png` | **Summed** rubric components over that episode |
| `judge_outcome_distributions.png` | **Box**: final HbA1c, FPG, return across N eval episodes |
| `judge_council_glucose_example.png` | (If council trace) one council glucose path |

**Run all cells top → bottom.** T4+ GPU for 4-bit. Edit **`CONFIG` only** to trade runtime vs. clearer separation of models. See section *3* for `judge_endpoint_episodes` (stochasticity panels).


### How the environment behaves (for reviewers)

- **Observation** each week: only a **subset** of patient state (week, HbA1c, FPG, BMI, BP, eGFR, flags) — see `DigitalTwinDiabetesEnv._obs`.
- **Action**: one JSON object per week (`noop`, `start`, `add`, …) parsed by `safe_action`; costs and side effects flow from the **twin** simulation.
- **Reward**: not a single number from a labeler — it is **decomposed** (glucose/HbA1c improvement, hypo, swing, cost, inactivity, time, cumulative hyper, safety tools, terminal). Inspect `reward/rubric.py` for **names** used in `judge_rubric_episode_totals.png`.
- **Stochasticity**: each `reset(seed=…)` draws a **patient**; training hammers many seeds. The **judge trajectories** optionally share one seed between random and distil for a **controlled** comparison; **outcome boxplots** always use **many** seeds.

```mermaid
flowchart LR
  obs[partial obs] --> policy[policy]
  policy --> act[JSON action]
  act --> twin[patient twin step]
  twin --> next[physiology + costs + SE]
  next --> rubric[rubric sum]
  rubric --> R[step return]
```

## 1. Clone repo and install
Set `YOUR_REPO_URL` and `CLONE_DIR` (monorepo: clone root may contain a `DART/` subfolder or be DART-only).

In [ ]:
import os, subprocess, sys
from pathlib import Path
from typing import Optional

YOUR_REPO_URL = "https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git"
CLONE_DIR = Path("/content/mano")
_M = Path("scripts") / "train_reinforce_twin.py"

def _dart_root(b: Path) -> Optional[Path]:
    if (b / "DART" / _M).is_file():
        return (b / "DART").resolve()
    if (b / _M).is_file():
        return b.resolve()
    return None

if not CLONE_DIR.is_dir():
    subprocess.run(["git", "clone", YOUR_REPO_URL, str(CLONE_DIR)], check=True)
R = _dart_root(CLONE_DIR)
if R is None and (CLONE_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--ff-only"], check=False)
    R = _dart_root(CLONE_DIR)
if R is None:
    raise FileNotFoundError(f"Cannot find {_M} under {CLONE_DIR}")
os.chdir(R)
os.environ["DART_REPO_ROOT"] = str(R)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-r", "requirements_hackathon.txt"], check=True)
print("DART root:", R)

## 2. GPU and Hugging Face
1. **Runtime → Change runtime type → T4 (or A100).**  
2. For gated Llama: paste a **read** token below (not committed to public GitHub).

In [ ]:
import os
from huggingface_hub import login

_P = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
os.environ["HF_TOKEN"] = _P
os.environ["MODEL_ID"] = "unsloth/Meta-Llama-3.1-8B-Instruct-unsloth-bnb-4bit"
if os.environ["HF_TOKEN"].strip() and os.environ["HF_TOKEN"] != _P:
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("model:", os.environ["MODEL_ID"])

## 3. `CONFIG` (only block to edit for speed/quality)
- **`judge_endpoint_episodes`**: how many **independent** eval episodes feed the **boxplots** (final HbA1c / FPG / return). Higher = smoother distributions, longer runtime.
- **`max_steps`**: horizon per episode (maps to env `max_steps`).
- Raise `small.updates` / `large.updates` for clearer training separation; lower for a quick smoke run.

In [ ]:
import os
from pathlib import Path

REPO = Path(os.environ["DART_REPO_ROOT"]).resolve()
os.chdir(REPO)
Path("logs").mkdir(exist_ok=True)
Path("docs/figures").mkdir(parents=True, exist_ok=True)

CONFIG = {
    "max_steps": 20,
    "random_episodes": 40,
    "small": {
        "model_id": "distilgpt2",
        "short_label": "distilgpt2",
        "updates": 24,
        "episodes_per_update": 2,
        "max_new_tokens": 48,
    },
    "large": {
        "model_id": os.environ.get("MODEL_ID", "unsloth/Meta-Llama-3.1-8B-Instruct-unsloth-bnb-4bit"),
        "short_label": "llama-8b-4bit",
        "updates": 8,
        "episodes_per_update": 1,
        "load_4bit": True,
        "trust_remote_code": True,
        "max_new_tokens": 40,
    },
    "council_episodes": 16,
    "train_seed_base": 101,
    "ma_window": 5,
    "bar_tail_episodes": 15,
    "judge_endpoint_episodes": 20,
    "judge_trace_env_seed": 50_200,
    "out_json": "logs/colab_experiment.json",
}

## 4. Train, save JSON, render plots
REINFORCE on `DigitalTwinDiabetesEnv`; log **traces** (1 example ep / policy, same patient seed for random vs distil) and **endpoints** (many seeds); then:

- `scripts/plot_colab_publication.py` — learning curves, bars, glucose, council
- `scripts/plot_colab_judge_insights.py` — clinical grid, rubric, actions, outcome boxplots

If there is no CUDA, the 8B block is skipped (random + distil + council only).

In [ ]:
import json, os, subprocess, sys, torch
from pathlib import Path

REPO = Path(os.environ["DART_REPO_ROOT"]).resolve()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from training.colab_episode_rl import (
    collect_council_clinical_trace,
    collect_endpoints_random_baseline,
    collect_endpoints_trained,
    collect_random_baseline,
    collect_trained_episode_glucose,
    council_self_repair_episode_log,
    rollout_clinical_trace_random,
    rollout_clinical_trace_trained,
    train_reinforce_with_episode_log,
)

M, sb = int(CONFIG["max_steps"]), int(CONFIG["train_seed_base"])
JSEED = int(CONFIG.get("judge_trace_env_seed", 50_200))
JEP = int(CONFIG.get("judge_endpoint_episodes", 20))

r_rows, _, g_random = collect_random_baseline(
    n_episodes=int(CONFIG["random_episodes"]), max_steps=M, seed=0, model_key="random"
)
# Same-patient one-episode trace for the random policy (for overlay vs trained)
trace_r = rollout_clinical_trace_random(env_seed=JSEED, max_steps=M, random_seed=0)
end_r = collect_endpoints_random_baseline(
    n_episodes=JEP, max_steps=M, seed=0, model_key="random"
)
S = CONFIG["small"]
s_rows, model_s, tok_s, dev_s = train_reinforce_with_episode_log(
    model_id=S["model_id"],
    short_label=S["short_label"],
    load_in_4bit=False,
    trust_remote_code=False,
    updates=int(S["updates"]),
    episodes_per_update=int(S["episodes_per_update"]),
    max_steps=M,
    train_seed_base=sb,
    max_new_tokens=int(S.get("max_new_tokens", 48)),
)
mtok = int(S.get("max_new_tokens", 48))
end_s = collect_endpoints_trained(
    model_s,
    tok_s,
    dev_s,
    n_episodes=JEP,
    max_steps=M,
    seed=3_000,
    model_key=S["short_label"],
    max_new_tokens=mtok,
)
# Same env seed as trace_r: apples-to-apples patient
trace_s = rollout_clinical_trace_trained(
    model_s,
    tok_s,
    dev_s,
    env_seed=JSEED,
    max_steps=M,
    max_new_tokens=mtok,
)
g_trained = collect_trained_episode_glucose(
    model_s, tok_s, dev_s, max_steps=M, max_new_tokens=mtok, env_seed=10_100
)
del model_s, tok_s
torch.cuda.empty_cache() if torch.cuda.is_available() else None

l_rows = []
if torch.cuda.is_available():
    L = CONFIG["large"]
    l_rows, m_l, t_l, _ = train_reinforce_with_episode_log(
        model_id=L["model_id"],
        short_label=L["short_label"],
        load_in_4bit=bool(L.get("load_4bit", True)),
        trust_remote_code=bool(L.get("trust_remote_code", True)),
        updates=int(L["updates"]),
        episodes_per_update=int(L["episodes_per_update"]),
        max_steps=M,
        train_seed_base=sb + 5_000,
        max_new_tokens=int(L.get("max_new_tokens", 40)),
    )
    del m_l, t_l
    torch.cuda.empty_cache()
else:
    print("(no GPU — skipped 4-bit 8B)")

c_rows, repair_marks = council_self_repair_episode_log(
    n_episodes=int(CONFIG["council_episodes"]), max_steps=M, seed=7
)
c_trace = collect_council_clinical_trace(
    env_seed=JSEED + 1, max_steps=M, council_seed=7
)

out_path = REPO / CONFIG["out_json"]
sl = S["short_label"]
payload = {
    "config": {k: v for k, v in CONFIG.items() if k != "out_json"},
    "episodes": r_rows + s_rows + l_rows + c_rows,
    "glucose": {"random": g_random, "trained": g_trained},
    "self_repair_episodes": repair_marks,
    "plot_order": ["random", "distilgpt2", "llama-8b-4bit"],
    "bar_models": ["random", "distilgpt2", "llama-8b-4bit"],
    "bar_tail_episodes": int(CONFIG.get("bar_tail_episodes", 15)),
    "ma_window": int(CONFIG.get("ma_window", 5)),
    "traces": {
        "random": trace_r,
        sl: trace_s,
        "council_self_repair": c_trace,
    },
    "endpoints": {"random": end_r, sl: end_s},
    "judge_trace_order": ["random", sl, "council_self_repair"],
    "judge_endpoint_order": ["random", sl],
}
out_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("saved", out_path)
ma = int(CONFIG["ma_window"])
subprocess.run(
    [
        sys.executable,
        "scripts/plot_colab_publication.py",
        "--in-json",
        str(out_path),
        "--out-dir",
        "docs/figures",
        "--ma-window",
        str(ma),
        "--also-svg",
    ],
    cwd=REPO,
    check=True,
)
subprocess.run(
    [sys.executable, "scripts/plot_colab_judge_insights.py", "--in-json", str(out_path), "--out-dir", "docs/figures", "--also-svg"],
    cwd=REPO,
    check=True,
)
print("figures → docs/figures/ (publication + judge_*)")

## 5. View figures in Colab

In [ ]:
import os
from pathlib import Path
from IPython.display import display, Image, Markdown

F = Path(os.environ["DART_REPO_ROOT"]) / "docs" / "figures"
pub = [
    "training_curve.png",
    "final_comparison_bars.png",
    "behavior_glucose.png",
    "self_repair_episodes.png",
]
judge = [
    "judge_clinical_state.png",
    "judge_step_and_cumulative_return.png",
    "judge_action_mix.png",
    "judge_rubric_episode_totals.png",
    "judge_outcome_distributions.png",
    "judge_council_glucose_example.png",
]
for title, group in (("A — Training / aggregate", pub), ("B — Environment & rubric (judge dashboard)", judge)):
    display(Markdown(f"**{title}**"))
    for name in group:
        p = F / name
        if p.is_file():
            display(Markdown(f"*{name}*"))
            display(Image(filename=str(p)))
        elif name in judge:
            pass
        else:
            display(Markdown(f"_(missing {name})_"))

## 6. Commit (optional, on your machine)
After download or `git pull`, add at least: `logs/colab_experiment.json` and the PNGs you care about from `docs/figures/`, including the **judge_*** panels if you use this notebook for evaluation.

To improve separation between **random / distil / 8B**, increase `CONFIG['small']['updates']` and `CONFIG['large']['updates']` and re-run from section 3. For **tighter** outcome boxplots, raise `judge_endpoint_episodes` (more CPU time).